In [ ]:
# Demo: LangChain + OpenAI Integration with LLMChain

from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

# 1. Initialize LLM (deterministic output)
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# 2. Create a prompt template
prompt = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} in simple terms suitable for a beginner."
)

# 3. Set up LLMChain
llm_chain = LLMChain(llm=llm, prompt=prompt)

# 4. Execute the chain
result = llm_chain.run("machine learning")
print("LLM Response:\n", result)


##  Prompt Templates with OpenAI + LangChain

In [12]:
# This script shows:
# 1. Creating reusable prompt templates
# 2. Injecting dynamic inputs
# 3. Running prompts with OpenAI models

# Install (if needed)
# pip install langchain openai tiktoken

from langchain.prompts import PromptTemplate
from openai import OpenAI

# ------------------------------------------------------------
# OpenAI client
# ------------------------------------------------------------
client = OpenAI(api_key="open-ai-api-key")


# ------------------------------------------------------------
# Step 1: Create a reusable prompt template
# ------------------------------------------------------------
summary_template = PromptTemplate(
    input_variables=["text"],
    template="""
You are a helpful AI assistant.

Summarize the following text in 3-4 concise bullet points:

Text:
{text}

Summary:
"""
)


# ------------------------------------------------------------
# Step 2: Format the prompt with dynamic input
# ------------------------------------------------------------
user_text = """
Artificial intelligence is transforming industries worldwide.
Companies use AI for automation, predictive analytics, and personalized user experiences.
The demand for AI engineers and data scientists continues to grow.
"""

prompt = summary_template.format(text=user_text)


# ------------------------------------------------------------
# Step 3: Call OpenAI model with the formatted prompt
# ------------------------------------------------------------
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print("\nGenerated Summary:\n")
print(response.choices[0].message.content)


# ------------------------------------------------------------
# Step 4: Another prompt template example (QA template)
# ------------------------------------------------------------
qa_template = PromptTemplate(
    input_variables=["context", "question"],
    template="""
Use the context to answer the question accurately.

Context:
{context}

Question:
{question}

Answer:
"""
)

context_text = "Python is an interpreted, high-level programming language used in AI, data science, automation, and web development."

question_text = "Why is Python preferred in AI?"

prompt2 = qa_template.format(
    context=context_text,
    question=question_text
)

response2 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt2}]
)

print("\nAnswer from QA Template:\n")
print(response2.choices[0].message.content)



Generated Summary:

- Artificial intelligence is revolutionizing industries globally.
- Companies are leveraging AI for automation, predictive analytics, and tailored user experiences.
- There is a growing demand for AI engineers and data scientists in the job market.

Answer from QA Template:

Python is preferred in AI due to its simplicity and readability, which facilitates easy learning and coding. Its vast ecosystem of libraries and frameworks, such as TensorFlow, PyTorch, and scikit-learn, provides powerful tools for machine learning and data manipulation. Additionally, Python's strong community support and extensive documentation make it easier for developers to find resources and collaborate on AI projects. Its versatility in handling various tasks across different domains also contributes to its popularity in AI development.


## Response Parsing / Structured Output Handling

In [14]:
# Goal:
# 1. Ask model to return clean JSON
# 2. Parse safely using Python
# 3. Handle malformed JSON
# ------------------------------------------------------------

# Requirements:
# pip install openai langchain tiktoken

from openai import OpenAI
from langchain.prompts import PromptTemplate
import json

client = OpenAI(api_key="OPENAI-API-KEY")

# ------------------------------------------------------------
# Step 1: Create structured output prompt
# ------------------------------------------------------------
json_template = PromptTemplate(
    input_variables=["text"],
    template="""
Extract key insights from the text below.
Return the result strictly in JSON with keys:
- "summary"
- "keywords" (as a list)
- "sentiment" (positive/neutral/negative)

Text:
{text}

Return only valid JSON. No explanations.
"""
)

# Sample input text
sample_text = """
Python is widely used for data science because of its rich ecosystem
of libraries such as NumPy, Pandas, and Scikit-learn.
It is also easy to learn and has a strong community.
"""

# Format prompt
prompt_json = json_template.format(text=sample_text)

# ------------------------------------------------------------
# Step 2: Call OpenAI model
# ------------------------------------------------------------
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt_json}]
)

raw_output = response.choices[0].message.content
print("Raw Model Output:\n", raw_output, "\n")

# ------------------------------------------------------------
# Step 3: Parse JSON safely
# ------------------------------------------------------------
try:
    parsed_output = json.loads(raw_output)
    print("Parsed JSON:\n", parsed_output)

    print("\nSummary:", parsed_output["summary"])
    print("Keywords:", parsed_output["keywords"])
    print("Sentiment:", parsed_output["sentiment"])

except json.JSONDecodeError:
    print("Model returned invalid JSON. Attempting to fix...")

    # Optional: cleaning hack (simple, not robust)
    cleaned = raw_output.strip().split("```")[-1]
    try:
        parsed_output = json.loads(cleaned)
        print("Fixed JSON:\n", parsed_output)
    except:
        print("Could not parse JSON. Manual review needed.")



Raw Model Output:
 ```json
{
  "summary": "Python is popular in data science due to its extensive libraries, ease of learning, and supportive community.",
  "keywords": ["Python", "data science", "libraries", "NumPy", "Pandas", "Scikit-learn", "easy to learn", "community"],
  "sentiment": "positive"
}
``` 

Model returned invalid JSON. Attempting to fix...
Could not parse JSON. Manual review needed.
